# 12 - External candidate structure universe (TRAIN + COCONUT + optional PubChem)

**Objective.** Build the candidate structure universe every C2 experiment retrieves from: one row per competition connectivity (tautomer-canonical InChIKey14) with provenance.

**Why it matters.** A hidden Class-2 molecule has no reference spectrum, so its structure can only come from an external database. A universe containing **only TRAIN structures** does not simulate Class 2 -- the hidden truth is just the TRAIN structure that lost its spectra. Section 5 states the universe status explicitly (`casmi.validation.c2_protocol`).

CPU / RAM heavy, no GPU. **Fully resumable**: every source chunk and key bucket writes a done-marker on Drive; a re-run skips finished work.

1. **Stage A** - each local source file (Drive `data/external/<source>/`, never downloaded by code) is read in chunks -> formula-mass prefilter -> the **training** canonicalizer `standardize_records` -> filters (mass 150-1200, organic, neutral, single component; rejected rows kept with reasons).
2. **Stage B** - per connectivity-key bucket, `merge.unify` deduplicates by `connectivity_key` (never by raw SMILES / database id / full InChIKey) and keeps provenance.
3. **Stage C** - `finalize_universe(universe_root, buckets)`: global key-sorted `candidate_id`, memory-mapped mass index, formula index, `universe_manifest.json` with the embedded source summary.

**Canonical layout** (universe root = `P.candidate_db_dir` = `candidates/`): `candidate_keys.npy`, `bucket_offsets.json`, `universe_manifest.json`, `index/`, `formula_index/`, `buckets/`, `variants/`; stage A under `standardized/ rejected/ filtered_out/ _done/`; reports under `manifests/`.

**Scale note.** RDKit tautomer canonicalization costs milliseconds per molecule per core: start with COCONUT and a curated PubChem subset (`configs/v6/pubchem_source.json`).

In [ ]:
# >>> CASMI BOOTSTRAP casmi-v2-bootstrap-2 (generated by scripts/stamp_notebooks.py; do not edit by hand)
# mount Drive -> env roots -> clone if missing, else fetch + checkout + pull --ff-only -> drop stale casmi modules -> src on sys.path
import importlib, os, subprocess, sys
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
REPO_ROOT = Path(os.environ.setdefault('ENVEDA_REPO_ROOT', '/content/Enveda'))
os.environ.setdefault('ENVEDA_SCRATCH_DIR', '/content/enveda_work')
BRANCH = os.environ.setdefault('ENVEDA_BRANCH', 'main')
SYNC_REPO = os.environ.get('ENVEDA_SYNC_REPO', '1' if IN_COLAB else '0') == '1'  # outside Colab the working tree is used as is
GITHUB_REPO = 'mybenkhadda/enveda-kaggle'


def _git(*args):
    r = subprocess.run(['git', '-C', str(REPO_ROOT), *args], capture_output=True, text=True)
    if r.returncode != 0:
        cmd = ' '.join(args)
        raise RuntimeError(f'git {cmd} failed in {REPO_ROOT}: {r.stderr.strip()} | Fix: discard local edits inside the clone '
                           f'(git -C {REPO_ROOT} stash) or delete {REPO_ROOT}, then re-run this cell; source lives on GitHub only.')
    return r.stdout.strip()


if SYNC_REPO:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')  # optional Colab secret (private repository)
        except Exception:
            token = None
    url = f'https://{token}@github.com/{GITHUB_REPO}.git' if token else f'https://github.com/{GITHUB_REPO}.git'
    if not (REPO_ROOT / '.git').exists():
        r = subprocess.run(['git', 'clone', '--branch', BRANCH, url, str(REPO_ROOT)], capture_output=True, text=True)
        if r.returncode != 0:
            err = r.stderr.replace(token, '***') if token else r.stderr
            raise RuntimeError(f'git clone of {GITHUB_REPO} failed: {err.strip()} | Fix: check the GITHUB_TOKEN Colab secret / network.')
    else:
        _git('fetch', 'origin', BRANCH)
        _git('checkout', BRANCH)
        _git('pull', '--ff-only', 'origin', BRANCH)
if not (REPO_ROOT / 'src' / 'casmi' / 'workspace' / 'versions.py').exists():
    raise RuntimeError(f'{REPO_ROOT} has no src/casmi/workspace/versions.py: the source is OLDER than this notebook. '
                       f'Fix: push the bootstrap/artifact-registry refactor to GitHub {BRANCH}, restart the runtime, re-run this cell.')
for _m in [m for m in list(sys.modules) if m == 'casmi' or m.startswith('casmi.')]:
    del sys.modules[_m]  # after a pull: never mix old and new source in one kernel
importlib.invalidate_caches()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.environ['ENVEDA_BOOTSTRAP_CELL'] = 'casmi-v2-bootstrap-2'
_head = _git('log', '-1', '--format=%h %s') if (REPO_ROOT / '.git').exists() else 'no git (uploaded copy)'
print(f'source {REPO_ROOT} @ {_head} | drive {DRIVE_ROOT} | colab {IN_COLAB} | synced {SYNC_REPO}')

In [ ]:
from casmi.workspace.bootstrap import CHEM_PACKAGES, bootstrap

NOTEBOOK = '12_colab_candidate_universe'
CTX = bootstrap(NOTEBOOK, notebook_api='casmi-v2-notebooks-4',
                uses_paths=('repo_root',),
                uses_artifacts=('candidate_staging_dir', 'universe_root', 'universe_buckets', 'candidate_mass_index', 'candidate_formula_index',
                                'candidate_manifests_dir', 'universe_source_summary', 'external_coconut_dir'),
                requires=('molecule_mass_variants', 'structure_table'),
                packages=CHEM_PACKAGES,
                signatures={'casmi.candidates.universe:standardize_source': ['source_cfg', 'out_dir', 'cfg'],
                            'casmi.candidates.universe:merge_bucket': ['bucket', 'std_root', 'mass_variants', 'out_root',
                                                                       'train_structure_table', 'cfg'],
                            'casmi.candidates.universe:finalize_universe': ['out_root', 'bucket_list', 'build_formula_index'],
                            'casmi.validation.c2_protocol:assess_universe': ['artifacts', 'cfg']})
CFG, P, ARTIFACTS, MON = CTX.cfg, CTX.paths, CTX.artifacts, CTX.monitor

In [ ]:
import json, time
import numpy as np, pandas as pd
from casmi.candidates.universe import UniverseBuildConfig

UCFG = UniverseBuildConfig.from_dict(CFG['universe'])
if UCFG.n_jobs in (None, -1):
    UCFG.n_jobs = os.cpu_count() or 1
WORK = ARTIFACTS.candidate_staging_dir   # stage A: standardized/, rejected/, filtered_out/, _done/
OUT = ARTIFACTS.universe_root            # stages B-C: canonical universe layout (same directory)
print(UCFG)
print('universe root:', OUT)

## 1. External sources

Every configured source (`universe.sources` in `configs/casmi_v2_colab.yaml`) with its expected Drive location. A missing file is reported and skipped, never an error -- but the universe is then TRAIN-only and section 5 marks the C2 protocol invalid.

In [ ]:
from casmi.candidates.sources import load_source_config
from casmi.workspace.config import external_source_specs

SOURCES = []
for spec in external_source_specs(CFG, P):
    scfg = load_source_config(spec['template_path'])
    scfg.path = str(spec['file'])
    SOURCES.append({'source': scfg.source, 'path': scfg.path, 'exists': spec['exists'],
                    'size_mb': round(spec['file'].stat().st_size / 1024 ** 2, 1) if spec['exists'] else None, 'cfg': scfg})
if SOURCES:
    display(pd.DataFrame(SOURCES).drop(columns='cfg'))
EXTERNAL_FILES_PRESENT = any(s['exists'] for s in SOURCES)
if not EXTERNAL_FILES_PRESENT:
    print('*** NO EXTERNAL SOURCE FILE ON DRIVE -> this run builds a TRAIN-ONLY universe: preliminary setup only, C2 protocol INVALID.')
    print('*** Expected COCONUT at', ARTIFACTS.external_coconut_dir / 'coconut.csv', '(columns: configs/v6/coconut_source.json)')

In [ ]:
# TRAIN structures keep their training contract (representative SMILES, mass variants)
from casmi.candidates.filters import train_filter_audit

MV = pd.read_parquet(ARTIFACTS.molecule_mass_variants)
ST = pd.read_parquet(ARTIFACTS.structure_table, columns=['smiles', 'connectivity_key', 'molecular_weight', 'formal_charge'])
train_one = MV.sort_values(['connectivity_key', 'mass_variant_id']).drop_duplicates('connectivity_key')
train_one = train_one.merge(ST.dropna(subset=['connectivity_key']).drop_duplicates('connectivity_key')[['connectivity_key', 'formal_charge']],
                            on='connectivity_key', how='left')
print(f'TRAIN: {len(MV):,} mass variants, {MV.connectivity_key.nunique():,} connectivities')
print('filters TRAIN WOULD fail (diagnostic only -- TRAIN is never filtered by default):')
display(train_filter_audit(train_one, UCFG.filters))

## 2. Stage A: standardize external sources (resumable per chunk)

In [ ]:
from casmi.candidates.universe import standardize_source

STAGE_A = []
for s in SOURCES:
    if not s['exists']:
        print('NOT PROVIDED YET', s['source'], s['path'])
        continue
    with MON.stage('stage A ' + s['source']):
        t0 = time.time()
        man = standardize_source(s['cfg'], WORK, UCFG)
        man['seconds'] = time.time() - t0
    STAGE_A.append(man)
STAGE_A = pd.concat(STAGE_A, ignore_index=True) if STAGE_A else pd.DataFrame()
if len(STAGE_A):
    display(STAGE_A.groupby('source')[['n_input', 'n_prefiltered', 'n_rejected', 'n_filtered', 'n_kept']].sum())
else:
    print('no external source processed -> the universe will contain TRAIN only')

## 3. Stage B: per-bucket deduplication by `connectivity_key` (resumable per bucket)

In [ ]:
from tqdm.auto import tqdm
from casmi.candidates.universe import all_buckets, merge_bucket, stage_a_signature

STD_ROOT = WORK / 'standardized'
BUCKETS = all_buckets(STD_ROOT, MV, UCFG.bucket_prefix_len)
ST_FOR_MERGE = ST.dropna(subset=['connectivity_key'])
STAGE_A_SIG = stage_a_signature(STD_ROOT, MV, UCFG)   # buckets built for other stage-A inputs (e.g. before COCONUT) are rebuilt
with MON.stage('stage B merge', n_items=len(BUCKETS), unit='buckets'):
    BREC = pd.DataFrame([merge_bucket(b, STD_ROOT, MV, OUT, ST_FOR_MERGE, UCFG, inputs_signature=STAGE_A_SIG)
                         for b in tqdm(BUCKETS, desc='buckets')])
print('stage-A inputs signature:', STAGE_A_SIG)
display(BREC.sort_values('n_connectivities', ascending=False).head(10))

## 4. Stage C: global candidate ids, mass index, formula index, manifest

Canonical call `finalize_universe(universe_root, buckets)`: everything (including `formula_index/` and `universe_manifest.json`) lands at the universe root, where every downstream loader expects it.

In [ ]:
from casmi.candidates.universe import finalize_universe

with MON.stage('stage C finalize'):
    MANIFEST = finalize_universe(OUT, BUCKETS)
print(json.dumps({k: MANIFEST[k] for k in ('n_candidates', 'n_buckets', 'universe_schema_version', 'manifest_version')}, indent=2))

## 5. Universe status: is this a Class-2 universe?

TRAIN-only / external-only / shared counts from the manifest. `C2_PROTOCOL_VALID` is True only if a non-TRAIN source contributed candidates (`c2_protocol` thresholds in the config). A TRAIN-only universe is valid for preliminary setup only.

In [ ]:
from casmi.validation.c2_protocol import assess_universe
from casmi.workspace.bootstrap import write_json

UNIVERSE = assess_universe(ARTIFACTS, CFG)
SOURCE_SUMMARY = MANIFEST['source_summary']
write_json(ARTIFACTS.universe_source_summary, {**SOURCE_SUMMARY, 'universe_status': UNIVERSE.universe_status, 'c2_mode': UNIVERSE.c2_mode,
                                                'c2_protocol_valid': UNIVERSE.c2_protocol_valid, 'reason': UNIVERSE.reason,
                                                'identity': UNIVERSE.identity, **CTX.metadata()})
OVERLAP = pd.DataFrame(sorted(SOURCE_SUMMARY['overlap'].items(), key=lambda kv: -kv[1]), columns=['sources', 'n_connectivities'])
display(pd.DataFrame([{k: SOURCE_SUMMARY[k] for k in ('n_candidates', 'train_only', 'external_only', 'train_and_external', 'no_source')}]))
display(OVERLAP)
print('UNIVERSE_STATUS', UNIVERSE.universe_status, '| EXTERNAL_SOURCE_PRESENT', UNIVERSE.external_source_present,
      '| C2_PROTOCOL_VALID', UNIVERSE.c2_protocol_valid, '| C2_MODE', UNIVERSE.c2_mode)
if not UNIVERSE.c2_protocol_valid:
    print('***', UNIVERSE.reason)

In [ ]:
from casmi.candidates.universe import rejected_summary

n_records = 0
for b in BUCKETS:
    n_records += int(pd.read_parquet(ARTIFACTS.universe_buckets / f'bucket={b}.parquet', columns=['n_source_records'])['n_source_records'].sum())
DEDUP = {'n_source_records_kept': n_records, 'n_unique_connectivities': SOURCE_SUMMARY['n_candidates'],
         'records_per_connectivity': n_records / max(SOURCE_SUMMARY['n_candidates'], 1)}
print(json.dumps(DEDUP, indent=2))
REJ = rejected_summary(WORK)
display(REJ.sort_values('n', ascending=False).head(30))

In [ ]:
import matplotlib.pyplot as plt
from casmi.candidates.formula_index import CompactFormulaIndex
from casmi.candidates.mass_index import CandidateMassIndex

MIDX = CandidateMassIndex.load(ARTIFACTS.candidate_mass_index)
FIDX = CompactFormulaIndex.load(ARTIFACTS.candidate_formula_index)
sample = np.asarray(MIDX.masses[:: max(1, len(MIDX) // 2_000_000)])   # strided sample of the mmap
fc = FIDX.counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(sample, bins=120); axes[0].set_xlabel('exact mass (Da)'); axes[0].set_title(f'mass distribution ({len(MIDX):,} variant rows)')
axes[1].hist(np.log10(fc), bins=60); axes[1].set_xlabel('log10 #candidates per formula'); axes[1].set_title(f'{len(FIDX):,} formulas')
fig.tight_layout()
ARTIFACTS.candidate_manifests_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(ARTIFACTS.candidate_manifests_dir / 'universe_distributions.png', dpi=120)
top = np.argsort(-fc)[:20]
display(pd.DataFrame({'formula': FIDX.vocab[top], 'n_candidates': fc[top]}))

## 6. LEAKAGE AUDIT

1. **Inputs**: TRAIN mass variants, the TRAIN structure table and user-supplied external databases only -- no spectrum, nothing from the visible competition test set.
2. **Identity**: one row per connectivity (keys strictly sorted and unique); every TRAIN connectivity present (truths are reachable unless deliberately removed for C3); the embedded source summary counts every candidate exactly once.
3. **Shortcut columns**: `has_reference_spectrum`, `in_train`, `train_present`, `candidate_sources`, ... are **metadata**, never C2 ranking features (`c2_protocol.TRAIN_PROVENANCE_COLUMNS`, `FORBIDDEN_SHORTCUT_FEATURES`); source flags such as `coconut_present` / `source_count` need an explicit provenance ablation before use.
4. **No labels** live in the universe; hidden truths are stripped of TRAIN provenance per fold at evaluation time.

In [ ]:
from casmi.candidates.provenance import FORBIDDEN_SHORTCUT_FEATURES
from casmi.candidates.universe import keys_to_ids, load_candidate_keys
from casmi.validation.c2_protocol import TRAIN_PROVENANCE_COLUMNS

UK = load_candidate_keys(OUT)
assert len(UK) == MANIFEST['n_candidates'] and (UK[:-1] < UK[1:]).all(), 'candidate keys must be strictly sorted and unique'
missing_train = int((keys_to_ids(UK, MV['connectivity_key'].astype(str).unique()) < 0).sum())
assert missing_train == 0, f'{missing_train} TRAIN connectivities missing from the universe'
assert sum(SOURCE_SUMMARY['overlap'].values()) == MANIFEST['n_candidates'], 'source summary does not cover every candidate exactly once'
assert {'has_reference_spectrum', 'in_train'} <= set(FORBIDDEN_SHORTCUT_FEATURES) and 'train_present' in TRAIN_PROVENANCE_COLUMNS
LEAKAGE_AUDIT = {'keys_sorted_unique': True, 'all_train_connectivities_present': True, 'source_summary_complete': True,
                 'forbidden_shortcut_features': list(FORBIDDEN_SHORTCUT_FEATURES), 'train_provenance_columns': list(TRAIN_PROVENANCE_COLUMNS),
                 'universe_status': UNIVERSE.universe_status, 'c2_protocol_valid': UNIVERSE.c2_protocol_valid}
print('LEAKAGE AUDIT:', json.dumps(LEAKAGE_AUDIT))

## 7. Persist

In [ ]:
from casmi.candidates.universe import UNIVERSE_SCHEMA_VERSION, V2_COLUMNS
from casmi.workspace.artifacts import write_metadata
from casmi.workspace.resources import memory_snapshot

write_metadata(OUT, NOTEBOOK, config={'universe': CFG['universe'], 'c2_protocol': CFG.get('c2_protocol')}, seed=CTX.seed,
               input_paths={'molecule_mass_variants': ARTIFACTS.molecule_mass_variants, 'structure_table': ARTIFACTS.structure_table,
                            **{s['source']: s['path'] for s in SOURCES}},
               schema_version=UNIVERSE_SCHEMA_VERSION, feature_list=V2_COLUMNS, repo_root=P.repo_root,
               extra={'manifest': MANIFEST, 'dedup': DEDUP, 'universe': UNIVERSE.as_dict(), 'leakage_audit': LEAKAGE_AUDIT,
                      'memory_at_end': memory_snapshot(), 'resources': MON.table().to_dict(orient='records'), **CTX.metadata()})
OVERLAP.to_parquet(ARTIFACTS.candidate_manifests_dir / 'source_overlap.parquet', index=False)
REJ.to_parquet(ARTIFACTS.candidate_manifests_dir / 'rejected_summary.parquet', index=False)
if len(STAGE_A):
    STAGE_A.to_parquet(ARTIFACTS.candidate_manifests_dir / 'stage_a_chunks.parquet', index=False)
print('universe saved under', OUT, '| status', UNIVERSE.universe_status)
if UNIVERSE.c2_protocol_valid:
    print('NEXT: 11_colab_hidden_like_validation with REBUILD = True, then 13_colab_candidate_recall')
else:
    print('NEXT: provide an external source (COCONUT at data/external/coconut/coconut.csv) and re-run this notebook;')
    print('      notebook 11 may be re-run now for setup, but its C2 regime stays PRELIMINARY and Gate A will be protocol-invalid.')